# 6. Inferensi — Analisis Sentimen dengan 3 Skema Model
**Kriteria:** Testing/Inferensi yang menghasilkan output kelas kategorikal

Notebook ini memuat semua model yang telah dilatih dan menjalankan prediksi pada teks baru.

| Skema | Algoritma | Fitur | Split |
|-------|-----------|-------|-------|
| 1 | SVM (LinearSVC) | TF-IDF unigram+bigram | 80/20 |
| 2 | LSTM | FastText 128-dim | 80/20 |
| 3 | BiLSTM + Attention | FastText 128-dim | 70/30 |

**Output:** Kelas kategorikal — `Negatif`, `Netral`, `Positif`

In [1]:
# Cell 1: Load all models
import pandas as pd
import numpy as np
import joblib
import re
import string
import os
import warnings
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Layer
import tensorflow.keras.backend as K
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
warnings.filterwarnings('ignore')

# ── Custom layer required to load Scheme 3 model ──
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], 1),
                                  initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='attention_bias', shape=(input_shape[1], 1),
                                  initializer='zeros', trainable=True)
        super(AttentionLayer, self).build(input_shape)
    def call(self, x):
        e = K.tanh(K.dot(x, self.W) + self.b)
        a = K.softmax(e, axis=1)
        return K.sum(x * a, axis=1)
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

# ── Load SVM models ──
tfidf = joblib.load('models/tfidf_vectorizer.pkl')
svm   = joblib.load('models/svm_model.pkl')

# ── Load tokenizer ──
tokenizer = joblib.load('models/tokenizer.pkl')

# ── Load deep learning models ──
model_lstm = load_model('models/lstm_model.h5')
model_attn = load_model('models/lstm_attn_model.h5',
                         custom_objects={'AttentionLayer': AttentionLayer})

print("Semua model berhasil dimuat!")
print(f"  ✓ SVM + TF-IDF vectorizer")
print(f"  ✓ LSTM model")
print(f"  ✓ BiLSTM+Attention model")

Semua model berhasil dimuat!
  ✓ SVM + TF-IDF vectorizer
  ✓ LSTM model
  ✓ BiLSTM+Attention model


In [2]:
# Cell 2: Preprocessing function for inference
# Must match the preprocessing pipeline used during training (2_preprocessing.ipynb)
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stop_words = set(stopwords.words('indonesian'))
stop_words.update(set(stopwords.words('english')))
custom_stopwords = {'bank', 'jago', 'aplikasi', 'app', 'nya', 'yang', 'di', 'ke',
                    'dan', 'ini', 'itu', 'dengan', 'untuk', 'dari', 'ada', 'saya',
                    'aku', 'kamu', 'anda', 'kita', 'mereka', 'bisa', 'sudah',
                    'akan', 'juga', 'atau', 'tapi', 'kalau', 'karena', 'saat',
                    'waktu', 'lagi', 'sama', 'masih', 'belum', 'mau', 'harus',
                    'iya', 'yaa', 'gak', 'na', 'sih', 'ku', 'ga', 'ya', 'gaa',
                    'loh', 'kah', 'woi', 'woii', 'woy'}
stop_words.update(custom_stopwords)

slang_dict = {
    'gk': 'tidak', 'ga': 'tidak', 'gak': 'tidak', 'ngga': 'tidak',
    'nggak': 'tidak', 'enggak': 'tidak', 'tdk': 'tidak', 'ndak': 'tidak',
    'kagak': 'tidak', 'kaga': 'tidak', 'gag': 'tidak',
    'yg': 'yang', 'dgn': 'dengan', 'dg': 'dengan', 'dngn': 'dengan',
    'utk': 'untuk', 'krn': 'karena', 'krna': 'karena', 'karna': 'karena',
    'sdh': 'sudah', 'udah': 'sudah', 'udh': 'sudah',
    'blm': 'belum', 'belom': 'belum', 'blum': 'belum',
    'sm': 'sama', 'jg': 'juga', 'jga': 'juga',
    'bgt': 'banget', 'bget': 'banget', 'bngt': 'banget', 'bet': 'banget',
    'dr': 'dari', 'dri': 'dari', 'bs': 'bisa', 'bsa': 'bisa',
    'lg': 'lagi', 'lgi': 'lagi', 'tp': 'tapi', 'tpi': 'tapi',
    'ttp': 'tetap', 'sy': 'saya', 'aq': 'saya', 'ak': 'aku',
    'gua': 'saya', 'gue': 'saya', 'gw': 'saya',
    'lo': 'kamu', 'lu': 'kamu', 'u': 'kamu', 'km': 'kamu',
    'mksd': 'maksud', 'bkn': 'bukan', 'krg': 'kurang',
    'cuma': 'hanya', 'emg': 'memang', 'emang': 'memang',
    'lbh': 'lebih', 'jd': 'jadi', 'msh': 'masih',
    'hrs': 'harus', 'skrg': 'sekarang', 'kmrn': 'kemarin',
    'mo': 'mau', 'nih': 'ini', 'tuh': 'itu',
    'app': 'aplikasi', 'apps': 'aplikasi',
    'tf': 'transfer', 'eror': 'error', 'err': 'error',
    'updt': 'perbarui', 'ftur': 'fitur',
    'bug': 'kesalahan', 'bugg': 'kesalahan',
    'bgus': 'bagus', 'ok': 'baik', 'oke': 'baik', 'bgs': 'bagus',
    'mantap': 'bagus', 'mantul': 'bagus', 'keren': 'bagus',
    'jelek': 'buruk', 'jlek': 'buruk', 'parah': 'buruk', 'payah': 'buruk',
    'lemot': 'lambat', 'lelet': 'lambat',
    'aja': 'saja', 'gimana': 'bagaimana', 'gmn': 'bagaimana',
    'kalo': 'kalau', 'tau': 'tahu', 'pake': 'pakai',
    'dapet': 'dapat', 'dpt': 'dapat', 'sampe': 'sampai',
    'abis': 'habis', 'bentar': 'sebentar', 'ntar': 'nanti',
    'makasih': 'terima kasih', 'thanks': 'terima kasih',
    'sorry': 'maaf', 'sory': 'maaf',
    'bnyk': 'banyak', 'byk': 'banyak', 'sdkt': 'sedikit',
    'sgt': 'sangat', 'lgsg': 'langsung', 'cpt': 'cepat',
    'btw': 'ngomong-ngomong', 'otw': 'sedang di jalan',
    'dll': 'dan lain-lain', 'gpp': 'tidak apa-apa',
}

def preprocess_for_inference(text):
    text = str(text).lower()
    text = re.sub(r'@[A-Za-z0-9]+', '', text)
    text = re.sub(r'#[A-Za-z0-9]+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'[0-9]+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [slang_dict.get(w, w) for w in words]
    tokens = word_tokenize(' '.join(words))
    tokens = [w for w in tokens if w not in stop_words and len(w) > 1]
    tokens = [stemmer.stem(w) for w in tokens]
    return ' '.join(tokens)

print("Fungsi preprocessing siap digunakan.")

Fungsi preprocessing siap digunakan.


In [3]:
# Cell 3: Prediction functions
MAX_LEN = 100
label_names = {0: 'Negatif', 1: 'Netral', 2: 'Positif'}

def predict_svm(texts):
    cleaned = [preprocess_for_inference(t) for t in texts]
    X_tfidf = tfidf.transform(cleaned)
    preds = svm.predict(X_tfidf)
    return [label_names[p] for p in preds]

def predict_lstm(texts):
    cleaned = [preprocess_for_inference(t) for t in texts]
    seqs = tokenizer.texts_to_sequences(cleaned)
    padded = pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')
    preds = np.argmax(model_lstm.predict(padded, verbose=0), axis=1)
    return [label_names[p] for p in preds]

def predict_lstm_attn(texts):
    cleaned = [preprocess_for_inference(t) for t in texts]
    seqs = tokenizer.texts_to_sequences(cleaned)
    padded = pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')
    preds = np.argmax(model_attn.predict(padded, verbose=0), axis=1)
    return [label_names[p] for p in preds]

print("Fungsi prediksi siap digunakan.")

Fungsi prediksi siap digunakan.


In [4]:
# Cell 4: Inference on sample texts
sample_texts = [
    "Aplikasi ini sangat bagus dan mudah digunakan, saya sangat puas",
    "Fitur lengkap dan tampilan menarik, transfer juga cepat dan aman",
    "Biasa saja, tidak ada yang istimewa dari aplikasi ini",
    "Kadang bisa kadang tidak, lumayan lah untuk kebutuhan sehari-hari",
    "Sangat lambat dan sering error, tolong diperbaiki segera",
    "Aplikasi sangat mengecewakan, uang saya tidak bisa ditarik",
    "Bagus sekali pelayanannya, customer service cepat merespons",
    "Sudah beberapa kali gagal transfer, sangat tidak memuaskan",
    "Mantap banget, fitur tabungan sangat membantu menabung rutin",
    "Tidak bisa login dari tadi, error terus menerus sangat kesal",
]

results = pd.DataFrame({
    'Teks Ulasan': sample_texts,
    'Skema 1 - SVM': predict_svm(sample_texts),
    'Skema 2 - LSTM': predict_lstm(sample_texts),
    'Skema 3 - BiLSTM+Attn': predict_lstm_attn(sample_texts),
})

pd.set_option('display.max_colwidth', 55)
print("=" * 80)
print("HASIL INFERENSI — ANALISIS SENTIMEN BANK JAGO")
print("=" * 80)
results

HASIL INFERENSI — ANALISIS SENTIMEN BANK JAGO


,Teks Ulasan,Skema 1 - SVM,Skema 2 - LSTM,Skema 3 - BiLSTM+Attn
0,"Aplikasi ini sangat bagus dan mudah digunakan, saya...",Positif,Positif,Positif
1,"Fitur lengkap dan tampilan menarik, transfer juga c...",Positif,Positif,Positif
2,"Biasa saja, tidak ada yang istimewa dari aplikasi ini",Positif,Negatif,Positif
3,"Kadang bisa kadang tidak, lumayan lah untuk kebutuh...",Positif,Negatif,Positif
4,"Sangat lambat dan sering error, tolong diperbaiki s...",Negatif,Negatif,Negatif
5,"Aplikasi sangat mengecewakan, uang saya tidak bisa ...",Positif,Negatif,Negatif
6,"Bagus sekali pelayanannya, customer service cepat m...",Positif,Negatif,Positif
7,"Sudah beberapa kali gagal transfer, sangat tidak me...",Negatif,Negatif,Negatif
8,"Mantap banget, fitur tabungan sangat membantu menab...",Positif,Positif,Positif
9,"Tidak bisa login dari tadi, error terus menerus san...",Negatif,Negatif,Negatif


In [5]:
# Cell 5: Interactive single prediction
def analyze_sentiment(text):
    """Predict sentiment for a single review using all 3 schemes."""
    print(f"\nTeks: '{text}'")
    print(f"  Skema 1 (SVM + TF-IDF)           : {predict_svm([text])[0]}")
    print(f"  Skema 2 (LSTM + FastText)         : {predict_lstm([text])[0]}")
    print(f"  Skema 3 (BiLSTM + Attention)      : {predict_lstm_attn([text])[0]}")
    print()

# Contoh inferensi
analyze_sentiment("Aplikasi jago sangat membantu kebutuhan finansial saya sehari-hari")
analyze_sentiment("Sering force close dan data tidak sinkron, sangat mengganggu sekali")
analyze_sentiment("Cukup baik tapi masih ada beberapa kekurangan yang perlu diperbaiki")


Teks: 'Aplikasi jago sangat membantu kebutuhan finansial saya sehari-hari'
  Skema 1 (SVM + TF-IDF)           : Positif
  Skema 2 (LSTM + FastText)         : Positif
  Skema 3 (BiLSTM + Attention)      : Positif


Teks: 'Sering force close dan data tidak sinkron, sangat mengganggu sekali'
  Skema 1 (SVM + TF-IDF)           : Negatif
  Skema 2 (LSTM + FastText)         : Negatif
  Skema 3 (BiLSTM + Attention)      : Negatif


Teks: 'Cukup baik tapi masih ada beberapa kekurangan yang perlu diperbaiki'
  Skema 1 (SVM + TF-IDF)           : Positif
  Skema 2 (LSTM + FastText)         : Negatif
  Skema 3 (BiLSTM + Attention)      : Positif



In [6]:
# Cell 6: Summary comparison of all 3 schemes
summary = pd.DataFrame({
    'Skema': ['Skema 1', 'Skema 2', 'Skema 3'],
    'Algoritma': ['SVM (LinearSVC)', 'LSTM', 'BiLSTM + Attention'],
    'Ekstraksi Fitur': ['TF-IDF (unigram+bigram)', 'FastText (128-dim)', 'FastText (128-dim)'],
    'Pembagian Data': ['80% / 20%', '80% / 20%', '70% / 30%'],
    'Target Akurasi': ['>= 85%', '>= 92%', '>= 92%'],
})

print("RINGKASAN 3 SKEMA PELATIHAN ANALISIS SENTIMEN")
print("=" * 70)
print(summary.to_string(index=False))
print()
print("Kelas Output: Negatif | Netral | Positif")

RINGKASAN 3 SKEMA PELATIHAN ANALISIS SENTIMEN
  Skema          Algoritma         Ekstraksi Fitur Pembagian Data Target Akurasi
Skema 1    SVM (LinearSVC) TF-IDF (unigram+bigram)      80% / 20%         >= 85%
Skema 2               LSTM      FastText (128-dim)      80% / 20%         >= 92%
Skema 3 BiLSTM + Attention      FastText (128-dim)      70% / 30%         >= 92%

Kelas Output: Negatif | Netral | Positif
